In [1]:
"""
Stage 3: Stratified Group Splitting for HAM10000

This notebook creates train/validation/test splits that:
1. Split by lesion_id (not image_id) to prevent data leakage
2. Use stratified sampling to preserve class distribution
3. Maintain 70%/15%/15% ratio

Why this matters:
- Some lesions have multiple images (same lesion in train+test = leakage)
- Class imbalance (67% nv vs 1% df) requires stratification
- Fixed random seed ensures reproducibility
"""

from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedShuffleSplit
import json

# Configuration
RANDOM_SEED = 42
TEST_SIZE = 0.15  # 15% for test
VAL_SIZE = 0.15  # 15% for validation

# Paths
PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data" / "raw"
SPLITS_DIR = PROJECT_ROOT / "data" / "splits"
SPLITS_DIR.mkdir(parents=True, exist_ok=True)

# Use clean metadata from validation stage
METADATA_PATH = DATA_DIR / "HAM10000_metadata_clean.csv"

print("=" * 60)
print("STAGE 3: STRATIFIED GROUP SPLITTING")
print("=" * 60)

STAGE 3: STRATIFIED GROUP SPLITTING


In [2]:
"""
Load the cleaned metadata and prepare for splitting.

We group by lesion_id because:
- Each lesion may have multiple images
- All images from same lesion must stay in same split
- This prevents data leakage
"""

# Load clean metadata
df = pd.read_csv(METADATA_PATH)
print(f"Loaded {len(df)} images from clean metadata")

# Group by lesion_id
# Each row = one lesion, with list of all its images
lesion_groups = df.groupby('lesion_id').agg({
    'image_id': list,           # All image IDs for this lesion
    'dx': 'first',              # Class label (same for all images)
    'age': 'first',
    'sex': 'first',
    'localization': 'first'
}).reset_index()

print(f"\nGrouped into {len(lesion_groups)} unique lesions")

# Check lesions with multiple images
multi_image = lesion_groups['image_id'].apply(len)
print(f"\nImages per lesion statistics:")
print(multi_image.describe())
print(f"\nLesions with multiple images: {(multi_image > 1).sum()}")

# Show class distribution at lesion level
print(f"\nClass distribution (lesion level):")
print(lesion_groups['dx'].value_counts())

Loaded 10015 images from clean metadata

Grouped into 7470 unique lesions

Images per lesion statistics:
count    7470.000000
mean        1.340696
std         0.633970
min         1.000000
25%         1.000000
50%         1.000000
75%         2.000000
max         6.000000
Name: image_id, dtype: float64

Lesions with multiple images: 1956

Class distribution (lesion level):
dx
nv       5403
bkl       727
mel       614
bcc       327
akiec     228
vasc       98
df         73
Name: count, dtype: int64


In [ ]:
"""
First split: Separate 15% for test set.

We use StratifiedShuffleSplit to:
1. Keep class proportions same as full dataset
2. Randomly shuffle while maintaining stratification
3. Use fixed random seed for reproducibility
"""

print("\n" + "=" * 60)
print("SPLIT 1: Creating Test Set (15%)")
print("=" * 60)

# StratifiedShuffleSplit ensures class distribution is preserved
sss_test = StratifiedShuffleSplit(
    n_splits=1,           # Only need one split
    test_size=TEST_SIZE,    # 15% for test
    random_state=RANDOM_SEED
)

trainval_idx, test_idx = next(
    sss_test.split(lesion_groups, lesion_groups['dx'])
)

trainval_lesions = lesion_groups.iloc[trainval_idx].copy()
test_lesions = lesion_groups.iloc[test_idx].copy()

print(f"Train+Val lesions: {len(trainval_lesions)} ({len(trainval_lesions)/len(lesion_groups)*100:.1f}%)")
print(f"Test lesions: {len(test_lesions)} ({len(test_lesions)/len(lesion_groups)*100:.1f}%)")

# Verify stratification
print("\nTest set class distribution:")
print(test_lesions['dx'].value_counts())    


SPLIT 1: Creating Test Set (15%)
Train+Val lesions: 6349 (85.0%)
Test lesions: 1121 (15.0%)

Test set class distribution:
dx
nv       811
bkl      109
mel       92
bcc       49
akiec     34
vasc      15
df        11
Name: count, dtype: int64


In [4]:
"""
Second split: Divide remaining 85% into train (70%) and validation (15%).

Val size calculation: 15% of total = 15%/85% of remaining = 17.6%
"""

print("\n" + "=" * 60)
print("SPLIT 2: Creating Train (70%) and Validation (15%)")
print("=" * 60)

# Calculate validation ratio from remaining data
# We have 85% left, want 15% of total, so 15/85
val_ratio = VAL_SIZE / (1 - TEST_SIZE)
print(f"Validation ratio from remaining: {val_ratio:.3f} ({VAL_SIZE}/{1-TEST_SIZE})")

sss_val = StratifiedShuffleSplit(
    n_splits=1,
    test_size=val_ratio,   # 17.6% of 85% = 15% of total
    random_state=RANDOM_SEED
)

# # Split trainval into train and validation
# for train_idx, val_idx in sss_val.split(trainval_lesions, trainval_lesions['dx']):
#     train_lesions = trainval_lesions.iloc[train_idx].copy()
#     val_lesions = trainval_lesions.iloc[val_idx].copy()

train_idx, val_idx = next(
    sss_val.split(trainval_lesions, trainval_lesions['dx'])
)

train_lesions = trainval_lesions.iloc[train_idx].copy()
val_lesions = trainval_lesions.iloc[val_idx].copy()

print(f"\nTrain lesions: {len(train_lesions)} ({len(train_lesions)/len(lesion_groups)*100:.1f}%)")
print(f"Val lesions: {len(val_lesions)} ({len(val_lesions)/len(lesion_groups)*100:.1f}%)")

print("\nValidation set class distribution:")
print(val_lesions['dx'].value_counts())


SPLIT 2: Creating Train (70%) and Validation (15%)
Validation ratio from remaining: 0.176 (0.15/0.85)

Train lesions: 5228 (70.0%)
Val lesions: 1121 (15.0%)

Validation set class distribution:
dx
nv       811
bkl      109
mel       92
bcc       49
akiec     34
vasc      15
df        11
Name: count, dtype: int64


In [5]:
"""
Convert lesion-level splits back to image-level dataframes.

Each lesion may have multiple images - we expand them all.
"""

def expand_lesions(lesion_df):
    """
    Convert lesion-level dataframe to image-level.
    
    Input: DataFrame with lesion_id and list of image_ids
    Output: DataFrame with one row per image
    """
    rows = []
    for _, row in lesion_df.iterrows():
        for img_id in row['image_id']:
            rows.append({
                'lesion_id': row['lesion_id'],
                'image_id': img_id,
                'dx': row['dx'],
                'age': row['age'],
                'sex': row['sex'],
                'localization': row['localization']
            })
    return pd.DataFrame(rows)

print("\n" + "=" * 60)
print("EXPANDING TO IMAGE LEVEL")
print("=" * 60)

train_df = expand_lesions(train_lesions)
val_df = expand_lesions(val_lesions)
test_df = expand_lesions(test_lesions)

print(f"Train images: {len(train_df)}")
print(f"Val images: {len(val_df)}")
print(f"Test images: {len(test_df)}")
print(f"Total: {len(train_df) + len(val_df) + len(test_df)}")

# Verify total matches
assert len(train_df) + len(val_df) + len(test_df) == len(df), "Split mismatch!"
print(f"\n✓ Total matches original: {len(df)}")


EXPANDING TO IMAGE LEVEL
Train images: 7054
Val images: 1464
Test images: 1497
Total: 10015

✓ Total matches original: 10015


In [6]:
"""
CRITICAL VERIFICATION: Ensure no lesion appears in multiple splits.

This is the most important check - data leakage would invalidate all results.
"""

print("\n" + "=" * 60)
print("CRITICAL: DATA LEAKAGE VERIFICATION")
print("=" * 60)

# Get unique lesion IDs in each split
train_lesion_ids = set(train_df['lesion_id'])
val_lesion_ids = set(val_df['lesion_id'])
test_lesion_ids = set(test_df['lesion_id'])

# Check for any overlaps
overlap_train_val = train_lesion_ids & val_lesion_ids
overlap_train_test = train_lesion_ids & test_lesion_ids
overlap_val_test = val_lesion_ids & test_lesion_ids

print(f"Train lesions: {len(train_lesion_ids)}")
print(f"Val lesions: {len(val_lesion_ids)}")
print(f"Test lesions: {len(test_lesion_ids)}")
print(f"Total unique: {len(train_lesion_ids | val_lesion_ids | test_lesion_ids)}")

print(f"\nOverlaps:")
print(f"  Train-Val: {len(overlap_train_val)}")
print(f"  Train-Test: {len(overlap_train_test)}")
print(f"  Val-Test: {len(overlap_val_test)}")

# CRITICAL: Fail if any overlap found
assert len(overlap_train_val) == 0, "DATA LEAKAGE: Train-Val overlap!"
assert len(overlap_train_test) == 0, "DATA LEAKAGE: Train-Test overlap!"
assert len(overlap_val_test) == 0, "DATA LEAKAGE: Val-Test overlap!"

print("\n" + "=" * 60)
print("✓ NO DATA LEAKAGE DETECTED - SPLIT IS VALID!")
print("=" * 60)


CRITICAL: DATA LEAKAGE VERIFICATION
Train lesions: 5228
Val lesions: 1121
Test lesions: 1121
Total unique: 7470

Overlaps:
  Train-Val: 0
  Train-Test: 0
  Val-Test: 0

✓ NO DATA LEAKAGE DETECTED - SPLIT IS VALID!


In [7]:
"""
Verify that class distributions are preserved across all splits.
"""

def print_class_distribution(df, name):
    """Print class distribution for a split."""
    dist = df['dx'].value_counts().sort_index()
    total = len(df)
    print(f"\n{name} ({total} images):")
    for cls, count in dist.items():
        pct = count / total * 100
        print(f"  {cls}: {count:5d} ({pct:5.1f}%)")
    return dist

print("\n" + "=" * 60)
print("CLASS DISTRIBUTION VERIFICATION")
print("=" * 60)

train_dist = print_class_distribution(train_df, "Train")
val_dist = print_class_distribution(val_df, "Validation")
test_dist = print_class_distribution(test_df, "Test")

# Compare with full dataset
print("\n" + "=" * 60)
print("CLASS PROPORTION COMPARISON")
print("=" * 60)
full_dist = df['dx'].value_counts(normalize=True).sort_index()

print(f"{'Class':<8} {'Full':<8} {'Train':<8} {'Val':<8} {'Test':<8}")
print("-" * 45)

for cls in full_dist.index:
    full_pct = full_dist[cls] * 100
    train_pct = train_dist.get(cls, 0) / len(train_df) * 100
    val_pct = val_dist.get(cls, 0) / len(val_df) * 100
    test_pct = test_dist.get(cls, 0) / len(test_df) * 100
    
    print(f"{cls:<8} {full_pct:>6.1f}% {train_pct:>6.1f}% {val_pct:>6.1f}% {test_pct:>6.1f}%")

print("\n✓ All splits have similar class distributions!")


CLASS DISTRIBUTION VERIFICATION

Train (7054 images):
  akiec:   233 (  3.3%)
  bcc:   365 (  5.2%)
  bkl:   775 ( 11.0%)
  df:    76 (  1.1%)
  mel:   777 ( 11.0%)
  nv:  4730 ( 67.1%)
  vasc:    98 (  1.4%)

Validation (1464 images):
  akiec:    43 (  2.9%)
  bcc:    72 (  4.9%)
  bkl:   167 ( 11.4%)
  df:    17 (  1.2%)
  mel:   168 ( 11.5%)
  nv:   975 ( 66.6%)
  vasc:    22 (  1.5%)

Test (1497 images):
  akiec:    51 (  3.4%)
  bcc:    77 (  5.1%)
  bkl:   157 ( 10.5%)
  df:    22 (  1.5%)
  mel:   168 ( 11.2%)
  nv:  1000 ( 66.8%)
  vasc:    22 (  1.5%)

CLASS PROPORTION COMPARISON
Class    Full     Train    Val      Test    
---------------------------------------------
akiec       3.3%    3.3%    2.9%    3.4%
bcc         5.1%    5.2%    4.9%    5.1%
bkl        11.0%   11.0%   11.4%   10.5%
df          1.1%    1.1%    1.2%    1.5%
mel        11.1%   11.0%   11.5%   11.2%
nv         66.9%   67.1%   66.6%   66.8%
vasc        1.4%    1.4%    1.5%    1.5%

✓ All splits have simila

In [8]:
"""
Save the splits as CSV files for use in training.

Also save configuration metadata for reproducibility.
"""

print("\n" + "=" * 60)
print("SAVING SPLIT MANIFESTS")
print("=" * 60)

# Save CSV files
train_df.to_csv(SPLITS_DIR / "train.csv", index=False)
val_df.to_csv(SPLITS_DIR / "val.csv", index=False)
test_df.to_csv(SPLITS_DIR / "test.csv", index=False)

print(f"✓ Train: {SPLITS_DIR / 'train.csv'} ({len(train_df)} rows)")
print(f"✓ Val:   {SPLITS_DIR / 'val.csv'} ({len(val_df)} rows)")
print(f"✓ Test:  {SPLITS_DIR / 'test.csv'} ({len(test_df)} rows)")

# Save split configuration
split_config = {
    "timestamp": pd.Timestamp.now().isoformat(),
    "random_seed": RANDOM_SEED,
    "test_size": TEST_SIZE,
    "val_size": VAL_SIZE,
    "train_size": 1 - TEST_SIZE - VAL_SIZE,
    
    "lesion_counts": {
        "total": len(lesion_groups),
        "train": len(train_lesions),
        "val": len(val_lesions),
        "test": len(test_lesions)
    },
    
    "image_counts": {
        "total": len(df),
        "train": len(train_df),
        "val": len(val_df),
        "test": len(test_df)
    },
    
    "class_distribution": {
        "full": df['dx'].value_counts().to_dict(),
        "train": train_dist.to_dict(),
        "val": val_dist.to_dict(),
        "test": test_dist.to_dict()
    },
    
    "verification": {
        "no_leakage": True,
        "stratified": True
    }
}

config_path = SPLITS_DIR / "split_config.json"
with open(config_path, "w") as f:
    json.dump(split_config, f, indent=2)

print(f"\n✓ Configuration: {config_path}")


SAVING SPLIT MANIFESTS
✓ Train: d:\skin_screening_system\data\splits\train.csv (7054 rows)
✓ Val:   d:\skin_screening_system\data\splits\val.csv (1464 rows)
✓ Test:  d:\skin_screening_system\data\splits\test.csv (1497 rows)

✓ Configuration: d:\skin_screening_system\data\splits\split_config.json


In [9]:
"""
Generate final summary of the splitting process.
"""

print("\n" + "=" * 60)
print("SPLITTING COMPLETE - SUMMARY")
print("=" * 60)

print(f"\nConfiguration:")
print(f"  Random seed: {RANDOM_SEED}")
print(f"  Split ratios: {1-TEST_SIZE-VAL_SIZE:.0%} / {VAL_SIZE:.0%} / {TEST_SIZE:.0%}")

print(f"\nSplit Statistics:")
print(f"  Total lesions: {len(lesion_groups):,}")
print(f"  Total images: {len(df):,}")
print(f"\n  Train: {len(train_lesions):,} lesions → {len(train_df):,} images ({len(train_df)/len(df):.1%})")
print(f"  Val:   {len(val_lesions):,} lesions → {len(val_df):,} images ({len(val_df)/len(df):.1%})")
print(f"  Test:  {len(test_lesions):,} lesions → {len(test_df):,} images ({len(test_df)/len(df):.1%})")

print(f"\nVerification:")
print(f"  ✓ No data leakage (lesion overlap: 0)")
print(f"  ✓ Stratified by class")
print(f"  ✓ All images assigned")

print(f"\nOutput files:")
for f in ['train.csv', 'val.csv', 'test.csv', 'split_config.json']:
    print(f"  {SPLITS_DIR / f}")

print("\n" + "=" * 60)
print("Ready for Stage 4: Data Loading")
print("=" * 60)


SPLITTING COMPLETE - SUMMARY

Configuration:
  Random seed: 42
  Split ratios: 70% / 15% / 15%

Split Statistics:
  Total lesions: 7,470
  Total images: 10,015

  Train: 5,228 lesions → 7,054 images (70.4%)
  Val:   1,121 lesions → 1,464 images (14.6%)
  Test:  1,121 lesions → 1,497 images (14.9%)

Verification:
  ✓ No data leakage (lesion overlap: 0)
  ✓ Stratified by class
  ✓ All images assigned

Output files:
  d:\skin_screening_system\data\splits\train.csv
  d:\skin_screening_system\data\splits\val.csv
  d:\skin_screening_system\data\splits\test.csv
  d:\skin_screening_system\data\splits\split_config.json

Ready for Stage 4: Data Loading
